In [ ]:
import os
import re
import ast
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "tripadvisor_european_restaurants.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "stefanoleone992/tripadvisor-european-restaurants",
  file_path,
)

print("First 5 records:", df.head())

Using Colab cache for faster access to the 'tripadvisor-european-restaurants' dataset.
First 5 records:        restaurant_link       restaurant_name                                                            original_location country               region      province                      city  \
0  g10001637-d10002227                Le 147  ["Europe", "France", "Nouvelle-Aquitaine", "Haute-Vienne", "Saint-Jouvent"]  France   Nouvelle-Aquitaine  Haute-Vienne             Saint-Jouvent   
1  g10001637-d14975787      Le Saint Jouvent  ["Europe", "France", "Nouvelle-Aquitaine", "Haute-Vienne", "Saint-Jouvent"]  France   Nouvelle-Aquitaine  Haute-Vienne             Saint-Jouvent   
2   g10002858-d4586832       Au Bout du Pont  ["Europe", "France", "Centre-Val de Loire", "Berry", "Indre", "Rivarennes"]  France  Centre-Val de Loire         Berry                Rivarennes   
3   g10002986-d3510044   Le Relais de Naiade             ["Europe", "France", "Nouvelle-Aquitaine", "Correze", "Lacelle"

In [ ]:
df

,restaurant_link,restaurant_name,original_location,country,region,province,city,address,latitude,longitude,claimed,awards,popularity_detailed,popularity_generic,top_tags,price_level,price_range,meals,cuisines,special_diets,features,vegetarian_friendly,vegan_options,gluten_free,original_open_hours,open_days_per_week,open_hours_per_week,working_shifts_per_week,avg_rating,total_reviews_count,default_language,reviews_count_in_default_language,excellent,very_good,average,poor,terrible,food,service,value,atmosphere,keywords
0,g10001637-d10002227,Le 147,"[""Europe"", ""France"", ""Nouvelle-Aquitaine"", ""Haute-Vienne"", ""Saint-Jouvent""]",France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,Claimed,NaN,#1 of 2 Restaurants in Saint-Jouvent,#1 of 2 places to eat in Saint-Jouvent,"Cheap Eats, French",€,NaN,"Lunch, Dinner",French,NaN,"Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service",N,N,N,NaN,NaN,NaN,NaN,4.0,36.0,English,2.0,2.0,0.0,0.0,0.0,0.0,4.0,4.5,4.0,NaN,NaN
1,g10001637-d14975787,Le Saint Jouvent,"[""Europe"", ""France"", ""Nouvelle-Aquitaine"", ""Haute-Vienne"", ""Saint-Jouvent""]",France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,Unclaimed,NaN,#2 of 2 Restaurants in Saint-Jouvent,#2 of 2 places to eat in Saint-Jouvent,Cheap Eats,€,NaN,NaN,NaN,NaN,NaN,N,N,N,NaN,NaN,NaN,NaN,4.0,5.0,All languages,5.0,2.0,2.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
2,g10002858-d4586832,Au Bout du Pont,"[""Europe"", ""France"", ""Centre-Val de Loire"", ""Berry"", ""Indre"", ""Rivarennes""]",France,Centre-Val de Loire,Berry,Rivarennes,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,Claimed,NaN,#1 of 1 Restaurant in Rivarennes,#1 of 1 places to eat in Rivarennes,"Cheap Eats, French, European",€,NaN,"Dinner, Lunch, Drinks","French, European",NaN,"Reservations, Seating, Table Service, Wheelchair Accessible",N,N,N,NaN,NaN,NaN,NaN,5.0,13.0,English,4.0,3.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
3,g10002986-d3510044,Le Relais de Naiade,"[""Europe"", ""France"", ""Nouvelle-Aquitaine"", ""Correze"", ""Lacelle""]",France,Nouvelle-Aquitaine,Correze,Lacelle,"9 avenue Porte de la Correze 19170, 19170 Lacelle France",45.642610,1.824460,Claimed,NaN,#1 of 1 Restaurant in Lacelle,#1 of 1 places to eat in Lacelle,"Cheap Eats, French",€,NaN,"Lunch, Dinner",French,NaN,"Reservations, Seating, Serves Alcohol, Table Service, Wheelchair Accessible",N,N,N,NaN,NaN,NaN,NaN,4.0,34.0,English,1.0,1.0,0.0,0.0,0.0,0.0,4.5,4.5,4.5,NaN,NaN
4,g10022428-d9767191,Relais Du MontSeigne,"[""Europe"", ""France"", ""Occitanie"", ""Aveyron"", ""Saint-Laurent-de-Levezou""]",France,Occitanie,Aveyron,Saint-Laurent-de-Levezou,"route du Montseigne, 12620 Saint-Laurent-de-Levezou France",44.208860,2.960470,Unclaimed,NaN,#1 of 1 Restaurant in Saint-Laurent-de-Levezou,#1 of 1 places to eat in Saint-Laurent-de-Levezou,"Mid-range, French",€€-€€€,NaN,"Lunch, Dinner",French,NaN,"Reservations, Seating, Wheelchair Accessible, Table Service",N,N,N,NaN,NaN,NaN,NaN,4.5,11.0,All languages,11.0,4.0,7.0,0.0,0.0,0.0,4.5,4.5,4.5,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1083392,g9710275-d10770782,Complex Popas Pacurari,"[""Europe"", ""Romania"", ""Northeast Romania"", ""Iasi County"", ""Valea Lupului""]",Romania,Northeast Romania,Iasi County,NaN,"Soseaua Pacurari, Valea Lupului 707410 Romania",47.172950,27.519110,Unclaimed,NaN,#1 of 1 Restaurant in Valea Lupului,#1 of 1 places to eat in Valea Lupului,NaN,NaN,NaN,"Lunch, Dinner",NaN,NaN,NaN,N,N,N,"{""Mon"": [""10:00-22:00""], ""Tue"": [""10:00-22:00""], ""Wed"": [""10:00-22:00""], ""Thu"": [""10:00-22:00""], ""Fri"": [""10:00-22:00""], ""Sat"": [""10:00-22:00""], ""Sun"": [""10:00-22:00""]}",7.0,84.0,7.0,2.5,2.0,English,1.0,0.0,0.0,0.0,0.0,1.0

In [ ]:
print("Numar total randuri:", len(df))
print("Numar coloane:", len(df.columns))
print("Duplicate complete:", df.duplicated().sum())

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_df = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)

display(missing_df.head(40))

Numar total randuri: 1083397
Numar coloane: 42
Duplicate complete: 0


,missing_count,missing_pct
keywords,984199,90.843800
atmosphere,821612,75.836651
awards,820264,75.712227
price_range,779070,71.909928
features,765990,70.702614
special_diets,743141,68.593600
open_days_per_week,489565,45.187960
open_hours_per_week,489565,45.187960
working_shifts_per_week,489565,45.187960
original_open_hours,489565,45.187960


In [ ]:
candidate_cols = [
    "restaurant_link",
    "restaurant_name",
    "original_location",
    "country",
    "region",
    "province",
    "city",
    "address",
    "latitude",
    "longitude",
    "popularity_detailed",
    "popularity_generic",
    "top_tags",
    "price_level",
    "price_range",
    "meals",
    "cuisines",
    "cuisine",
    "features",
    "special_diets",
    "avg_rating",
    "food",
    "service",
    "value",
    "atmosphere",
    "awards",
    "original_open_hours",
    "open_days_per_week",
    "open_hours_per_week",
    "working_shifts_per_week",
]

existing_cols = [c for c in candidate_cols if c in df.columns]
missing_cols = [c for c in candidate_cols if c not in df.columns]

print("Coloane gasite:", existing_cols)
print("Coloane lipsa in dataset:", missing_cols)

df_clean = df[existing_cols].copy()
print("Shape dupa selectia coloanelor:", df_clean.shape)

Coloane gasite: ['restaurant_link', 'restaurant_name', 'original_location', 'country', 'region', 'province', 'city', 'address', 'latitude', 'longitude', 'popularity_detailed', 'popularity_generic', 'top_tags', 'price_level', 'price_range', 'meals', 'cuisines', 'features', 'special_diets', 'avg_rating', 'food', 'service', 'value', 'atmosphere', 'awards', 'original_open_hours', 'open_days_per_week', 'open_hours_per_week', 'working_shifts_per_week']
Coloane lipsa in dataset: ['cuisine']
Shape dupa selectia coloanelor: (1083397, 29)


In [ ]:
def clean_text_basic(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    return s if s and s.lower() not in {"nan", "none", "null"} else np.nan


def safe_value(x, default="Unknown"):
    if pd.isna(x):
        return default
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return default
    return s


def normalize_text_lower(x):
    x = clean_text_basic(x)
    return x.lower() if pd.notna(x) else np.nan


def normalize_for_match(x):
    x = clean_text_basic(x)
    if pd.isna(x):
        return ""
    s = str(x).lower()
    s = re.sub(r"[^a-zA-Z0-9À-ÿ\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def parse_list_field(x):
    if pd.isna(x):
        return []

    values = None

    if isinstance(x, (list, tuple, set)):
        values = list(x)
    else:
        s = str(x).strip()
        if not s or s.lower() in {"nan", "none", "null", "unknown"}:
            return []

        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    values = list(parsed)
                else:
                    values = [s]
            except Exception:
                values = None

        if values is None:
            values = re.split(r"[,|;/]", s)

    cleaned = []
    seen = set()

    for item in values:
        item_clean = clean_text_basic(item)
        if pd.notna(item_clean):
            key = item_clean.lower()
            if key not in seen:
                cleaned.append(item_clean)
                seen.add(key)

    return cleaned

In [ ]:
object_cols = df_clean.select_dtypes(include="object").columns.tolist()

for col in object_cols:
    df_clean[col] = df_clean[col].apply(clean_text_basic)

print("Curatarea textuala de baza a fost aplicata.")
df_clean.head()

Curatarea textuala de baza a fost aplicata.


,restaurant_link,restaurant_name,original_location,country,region,province,city,address,latitude,longitude,popularity_detailed,popularity_generic,top_tags,price_level,price_range,meals,cuisines,features,special_diets,avg_rating,food,service,value,atmosphere,awards,original_open_hours,open_days_per_week,open_hours_per_week,working_shifts_per_week
0,g10001637-d10002227,Le 147,"[""Europe"", ""France"", ""Nouvelle-Aquitaine"", ""Haute-Vienne"", ""Saint-Jouvent""]",France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,#1 of 2 Restaurants in Saint-Jouvent,#1 of 2 places to eat in Saint-Jouvent,"Cheap Eats, French",€,NaN,"Lunch, Dinner",French,"Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service",NaN,4.0,4.0,4.5,4.0,NaN,NaN,NaN,NaN,NaN,NaN
1,g10001637-d14975787,Le Saint Jouvent,"[""Europe"", ""France"", ""Nouvelle-Aquitaine"", ""Haute-Vienne"", ""Saint-Jouvent""]",France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,#2 of 2 Restaurants in Saint-Jouvent,#2 of 2 places to eat in Saint-Jouvent,Cheap Eats,€,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,g10002858-d4586832,Au Bout du Pont,"[""Europe"", ""France"", ""Centre-Val de Loire"", ""Berry"", ""Indre"", ""Rivarennes""]",France,Centre-Val de Loire,Berry,Rivarennes,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,#1 of 1 Restaurant in Rivarennes,#1 of 1 places to eat in Rivarennes,"Cheap Eats, French, European",€,NaN,"Dinner, Lunch, Drinks","French, European","Reservations, Seating, Table Service, Wheelchair Accessible",NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,g10002986-d3510044,Le Relais de Naiade,"[""Europe"", ""France"", ""Nouvelle-Aquitaine"", ""Correze"", ""Lacelle""]",France,Nouvelle-Aquitaine,Correze,Lacelle,"9 avenue Porte de la Correze 19170, 19170 Lacelle France",45.642610,1.824460,#1 of 1 Restaurant in Lacelle,#1 of 1 places to eat in Lacelle,"Cheap Eats, French",€,NaN,"Lunch, Dinner",French,"Reservations, Seating, Serves Alcohol, Table Service, Wheelchair Accessible",NaN,4.0,4.5,4.5,4.5,NaN,NaN,NaN,NaN,NaN,NaN
4,g10022428-d9767191,Relais Du MontSeigne,"[""Europe"", ""France"", ""Occitanie"", ""Aveyron"", ""Saint-Laurent-de-Levezou""]",France,Occitanie,Aveyron,Saint-Laurent-de-Levezou,"route du Montseigne, 12620 Saint-Laurent-de-Levezou France",44.208860,2.960470,#1 of 1 Restaurant in Saint-Laurent-de-Levezou,#1 of 1 places to eat in Saint-Laurent-de-Levezou,"Mid-range, French",€€-€€€,NaN,"Lunch, Dinner",French,"Reservations, Seating, Wheelchair Accessible, Table Service",NaN,4.5,4.5,4.5,4.5,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:

rename_map = {
    "restaurant_link": "restaurant_id",
    "restaurant_name": "name",
    "avg_rating": "rating",
}

df_clean = df_clean.rename(columns={k: v for k, v in rename_map.items() if k in df_clean.columns})

required_safe_cols = [
    "restaurant_id",
    "name",
    "original_location",
    "country",
    "region",
    "province",
    "city",
    "address",
    "popularity_detailed",
    "popularity_generic",
    "top_tags",
    "price_level",
    "price_range",
    "meals",
    "features",
    "special_diets",
    "rating",
]

for col in required_safe_cols:
    if col not in df_clean.columns:
        df_clean[col] = np.nan

print(df_clean.columns.tolist())

['restaurant_id', 'name', 'original_location', 'country', 'region', 'province', 'city', 'address', 'latitude', 'longitude', 'popularity_detailed', 'popularity_generic', 'top_tags', 'price_level', 'price_range', 'meals', 'cuisines', 'features', 'special_diets', 'rating', 'food', 'service', 'value', 'atmosphere', 'awards', 'original_open_hours', 'open_days_per_week', 'open_hours_per_week', 'working_shifts_per_week']


In [ ]:

before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Duplicate complete eliminate: {before - len(df_clean)}")

if df_clean["restaurant_id"].notna().any():
    print("restaurant_id null:", df_clean["restaurant_id"].isna().sum())
    print("restaurant_id duplicate:", df_clean["restaurant_id"].duplicated().sum())

    before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=["restaurant_id"]).reset_index(drop=True)
    print(f"Duplicate pe restaurant_id eliminate: {before - len(df_clean)}")
else:
    print("Nu exista restaurant_id valid; se pastreaza deduplicarea completa.")

print("Shape dupa deduplicare:", df_clean.shape)

Duplicate complete eliminate: 0
restaurant_id null: 0
restaurant_id duplicate: 0
Duplicate pe restaurant_id eliminate: 0
Shape dupa deduplicare: (1083397, 29)


In [ ]:
numeric_cols = [
    "rating",
    "food",
    "service",
    "value",
    "atmosphere",
    "latitude",
    "longitude",
    "open_days_per_week",
    "open_hours_per_week",
    "working_shifts_per_week",
]

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print("Rating describe:")
display(df_clean["rating"].describe())

Rating describe:


,rating
count,986761.000000
mean,4.035943
std,0.713694
min,1.000000
25%,3.500000
50%,4.000000
75%,4.500000
max,5.000000


In [ ]:
for col in ["country", "region", "province", "city", "price_level", "price_range"]:
    if col in df_clean.columns:
        print(f"\nTop 15 valori pentru {col}:")
        print(df_clean[col].value_counts(dropna=False).head(15))


Top 15 valori pentru country:
country
Italy              224763
Spain              157479
France             155288
England            144681
Germany            115333
Greece              33763
Portugal            32592
The Netherlands     29792
Poland              24698
Belgium             23711
Austria             20487
Sweden              18555
Czech Republic      14844
Scotland            14215
Ireland             11203
Name: count, dtype: int64

Top 15 valori pentru region:
region
NaN                           50323
Lombardy                      33097
Ile-de-France                 31271
Andalucia                     29562
Catalonia                     28569
Lazio                         23831
London                        22942
Bavaria                       21531
North Rhine-Westphalia        21116
Auvergne-Rhone-Alpes          20753
Provence-Alpes-Cote d'Azur    19925
Tuscany                       18861
Valencian Country             18406
Community of Madrid           18130
Vene

In [ ]:
for col in ["country", "region", "province", "city"]:
    df_clean[f"{col}_norm"] = df_clean[col].apply(normalize_text_lower)

df_clean[
    [
        "country",
        "country_norm",
        "region",
        "region_norm",
        "province",
        "province_norm",
        "city",
        "city_norm",
    ]
].head()

,country,country_norm,region,region_norm,province,province_norm,city,city_norm
0,France,france,Nouvelle-Aquitaine,nouvelle-aquitaine,Haute-Vienne,haute-vienne,Saint-Jouvent,saint-jouvent
1,France,france,Nouvelle-Aquitaine,nouvelle-aquitaine,Haute-Vienne,haute-vienne,Saint-Jouvent,saint-jouvent
2,France,france,Centre-Val de Loire,centre-val de loire,Berry,berry,Rivarennes,rivarennes
3,France,france,Nouvelle-Aquitaine,nouvelle-aquitaine,Correze,correze,Lacelle,lacelle
4,France,france,Occitanie,occitanie,Aveyron,aveyron,Saint-Laurent-de-Levezou,saint-laurent-de-levezou


In [ ]:
LOCATION_STOPWORDS = {
    "europe",
    "restaurants",
    "restaurant",
    "places to eat",
    "things to do",
}


def parse_original_location(x):
    values = parse_list_field(x)
    return [v for v in values if pd.notna(clean_text_basic(v))]


def extract_city_from_original_location(row):
    loc = parse_original_location(row.get("original_location"))

    if not loc:
        return np.nan

    country = normalize_for_match(row.get("country"))
    region = normalize_for_match(row.get("region"))
    province = normalize_for_match(row.get("province"))

    blocked = {country, region, province, "", "europe"}

    candidates = []

    for item in loc:
        item_norm = normalize_for_match(item)

        if not item_norm or item_norm in blocked or item_norm in LOCATION_STOPWORDS:
            continue

        candidates.append(item)

    if candidates:
        return candidates[-1]

    return np.nan


def extract_city_from_popularity(x):
    s = clean_text_basic(x)

    if pd.isna(s):
        return np.nan

    patterns = [
        r"#?\s*\d+\s+of\s+\d+\s+restaurants?\s+in\s+(.+)$",
        r"#?\s*\d+\s+of\s+\d+\s+places?\s+to\s+eat\s+in\s+(.+)$",
        r"restaurants?\s+in\s+(.+)$",
        r"places?\s+to\s+eat\s+in\s+(.+)$",
    ]

    for pattern in patterns:
        m = re.search(pattern, str(s), flags=re.IGNORECASE)

        if m:
            city = clean_text_basic(m.group(1))

            if pd.notna(city):
                city = re.sub(r"\s+restaurants?$", "", city, flags=re.IGNORECASE).strip()
                return city if city else np.nan

    return np.nan


def choose_city(row):
    city = clean_text_basic(row.get("city"))

    if pd.notna(city):
        return city, "original_city"

    city = extract_city_from_original_location(row)

    if pd.notna(city):
        return city, "original_location"

    city = extract_city_from_popularity(row.get("popularity_detailed"))

    if pd.notna(city):
        return city, "popularity_detailed"

    city = extract_city_from_popularity(row.get("popularity_generic"))

    if pd.notna(city):
        return city, "popularity_generic"

    return np.nan, "missing"


city_info = df_clean.apply(choose_city, axis=1)

df_clean["city_filled"] = city_info.apply(lambda x: x[0])
df_clean["city_source"] = city_info.apply(lambda x: x[1])
df_clean["city_filled_norm"] = df_clean["city_filled"].apply(normalize_text_lower)

df_clean["city_from_address"] = np.nan

print(df_clean["city_source"].value_counts(dropna=False))
display(
    df_clean[
        ["city", "original_location", "popularity_detailed", "city_filled", "city_source"]
    ].sample(10, random_state=42)
)

city_source
original_city          682712
original_location      383551
popularity_detailed     13435
missing                  2007
popularity_generic       1692
Name: count, dtype: int64


,city,original_location,popularity_detailed,city_filled,city_source
962380,Athens,"[""Europe"", ""Greece"", ""Attica"", ""Athens""]",#2108 of 2303 Restaurants in Athens,Athens,original_city
746926,NaN,"[""Europe"", ""Italy"", ""Trentino-Alto Adige"", ""Province of Trento"", ""Trento""]",#78 of 354 Restaurants in Trento,Trento,original_location
556479,Cardiff,"[""Europe"", ""United Kingdom (UK)"", ""Wales"", ""Southern Wales"", ""Cardiff""]",#713 of 817 Restaurants in Cardiff,Cardiff,original_city
511831,Liverpool,"[""Europe"", ""United Kingdom (UK)"", ""England"", ""Merseyside"", ""Liverpool""]",NaN,Liverpool,original_city
76556,Nice,"[""Europe"", ""France"", ""Provence-Alpes-Cote d'Azur"", ""French Riviera - Cote d'Azur"", ""Nice""]",#128 of 1657 Restaurants in Nice,Nice,original_city
1066071,Copenhagen,"[""Europe"", ""Denmark"", ""Zealand"", ""Copenhagen""]",#946 of 1969 Restaurants in Copenhagen,Copenhagen,original_city
184905,Noordwijkerhout,"[""Europe"", ""The Netherlands"", ""South Holland Province"", ""Noordwijkerhout""]",NaN,Noordwijkerhout,original_city
908104,NaN,"[""Europe"", ""Poland"", ""Central Poland"", ""Lodz Province"", ""Zdunska Wola""]",#11 of 13 Restaurants in Zdunska Wola,Zdunska Wola,original_location
48319,Paris,"[""Europe"", ""France"", ""Ile-de-France"", ""Paris""]",NaN,Paris,original_city
674030,NaN,"[""Europe"", ""Italy"", ""Piedmont"", ""Province of Vercelli"", ""Vercelli""]",#85 of 114 Restaurants in Vercelli,Vercelli,original_location


In [ ]:
suspicious_city_values = {
    "street",
    "st",
    "road",
    "rd",
    "avenue",
    "ave",
    "square",
    "center",
    "centre",
    "park",
    "hotel",
    "restaurant",
    "restaurants",
    "best",
    "cheap",
    "food",
    "pub",
    "bar",
    "cafe",
    "cafes",
    "europe",
    "united kingdom uk",
    "united kingdom (uk)",
    "united kingdom",
    "uk",
    "england",
    "scotland",
    "wales",
    "northern ireland",
}


def compute_city_suspicious_mask(dataframe):
    city_norm = dataframe["city_filled"].apply(normalize_for_match)
    country_norm = dataframe["country"].apply(normalize_for_match)
    region_norm = dataframe["region"].apply(normalize_for_match)
    province_norm = dataframe["province"].apply(normalize_for_match)

    city_is_known_bad = city_norm.isin(suspicious_city_values)

    city_equals_country = (
        city_norm.notna()
        & country_norm.notna()
        & (city_norm == country_norm)
        & (city_norm != "")
    )

    city_equals_region = (
        city_norm.notna()
        & region_norm.notna()
        & (city_norm == region_norm)
        & (city_norm != "")
    )

    city_equals_province = (
        city_norm.notna()
        & province_norm.notna()
        & (city_norm == province_norm)
        & (city_norm != "")
    )

    return city_is_known_bad | city_equals_country | city_equals_region | city_equals_province


initial_suspicious_mask = compute_city_suspicious_mask(df_clean)

print("Orase suspecte initial:", int(initial_suspicious_mask.sum()))

if initial_suspicious_mask.sum() > 0:
    print("\nTop city_filled suspecte initial:")
    display(
        df_clean.loc[initial_suspicious_mask, "city_filled"]
        .value_counts(dropna=False)
        .head(30)
    )


def repair_city_from_popularity(row):
    city = extract_city_from_popularity(row.get("popularity_detailed"))
    if pd.notna(city):
        return city, "popularity_detailed_repair"

    city = extract_city_from_popularity(row.get("popularity_generic"))
    if pd.notna(city):
        return city, "popularity_generic_repair"

    return row.get("city_filled"), row.get("city_source")


repair_info = df_clean.loc[initial_suspicious_mask].apply(repair_city_from_popularity, axis=1)

df_clean.loc[initial_suspicious_mask, "city_filled"] = repair_info.apply(lambda x: x[0])
df_clean.loc[initial_suspicious_mask, "city_source"] = repair_info.apply(lambda x: x[1])
df_clean["city_filled_norm"] = df_clean["city_filled"].apply(normalize_text_lower)

df_clean["city_is_suspicious"] = compute_city_suspicious_mask(df_clean)

print("\nOrase suspecte dupa repair:", int(df_clean["city_is_suspicious"].sum()))

print("\nSurse city dupa repair:")
display(df_clean["city_source"].value_counts(dropna=False).head(20))

print("\nTop city_filled dupa repair:")
display(df_clean["city_filled"].value_counts(dropna=False).head(30))

if df_clean["city_is_suspicious"].sum() > 0:
    print("\nExemple ramase suspecte:")
    display(
        df_clean.loc[
            df_clean["city_is_suspicious"],
            [
                "name",
                "country",
                "region",
                "province",
                "city",
                "city_filled",
                "city_source",
                "original_location",
                "popularity_detailed",
                "popularity_generic",
            ],
        ].head(50)
    )

    print("\nTop city_filled ramase suspecte:")
    display(
        df_clean.loc[df_clean["city_is_suspicious"], "city_filled"]
        .value_counts(dropna=False)
        .head(30)
    )

Orase suspecte initial: 46831

Top city_filled suspecte initial:


,count
city_filled,
United Kingdom (UK),31063
Berlin,6530
Hamburg,3218
Brussels,2515
Bucharest,2006
Durham,284
Melilla,176
Stirling,174
Ceuta,108



Orase suspecte dupa repair: 46831

Surse city dupa repair:


,count
city_source,
original_city,682121
original_location,355991
popularity_detailed_repair,36832
popularity_generic_repair,6430
missing,2007
popularity_detailed,16



Top city_filled dupa repair:


,count
city_filled,
London,20057
Paris,18129
Rome,12603
Madrid,12134
Barcelona,10285
Milan,8382
Berlin,6530
Prague,6035
Lisbon,5261



Exemple ramase suspecte:


,name,country,region,province,city,city_filled,city_source,original_location,popularity_detailed,popularity_generic
10055,Restaurant du Lac de Baudreix,France,Nouvelle-Aquitaine,NaN,NaN,Nouvelle-Aquitaine,popularity_detailed_repair,"[""Europe"", ""France"", ""Nouvelle-Aquitaine""]",#2 of 2 Restaurants in Nouvelle-Aquitaine,NaN
10056,Le Palais d'Orlac,France,Nouvelle-Aquitaine,NaN,NaN,Nouvelle-Aquitaine,popularity_detailed_repair,"[""Europe"", ""France"", ""Nouvelle-Aquitaine""]",#1 of 2 Restaurants in Nouvelle-Aquitaine,NaN
10057,L'Escapade,France,Grand Est,NaN,NaN,Grand Est,popularity_detailed_repair,"[""Europe"", ""France"", ""Grand Est""]",#2 of 4 Restaurants in Grand Est,NaN
10058,L'Escale du Ried,France,Grand Est,NaN,NaN,Grand Est,popularity_detailed_repair,"[""Europe"", ""France"", ""Grand Est""]",#1 of 4 Restaurants in Grand Est,NaN
10059,Au P'tit Troc,France,Grand Est,NaN,NaN,Grand Est,popularity_detailed_repair,"[""Europe"", ""France"", ""Grand Est""]",#3 of 4 Restaurants in Grand Est,NaN
10060,Aux Deux Clefs Restaurant,France,Grand Est,NaN,NaN,Grand Est,popularity_detailed_repair,"[""Europe"", ""France"", ""Grand Est""]",#4 of 4 Restaurants in Grand Est,NaN
10061,Auberge Catalane,France,Occitanie,NaN,NaN,Occitanie,popularity_detailed_repair,"[""Europe"", ""France"", ""Occitanie""]",#5 of 8 Restaurants in Occitanie,NaN
10062,Fabre Christian,France,Occitanie,NaN,NaN,Occitanie,popularity_detailed_repair,"[""Europe"", ""France"", ""Occitanie""]",#2 of 8 Restaurants in Occitanie,NaN
10063,Le Catharome,France,Occitanie,NaN,NaN,Occitanie,popularity_detailed_repair,"[""Europe"", ""France"", ""Occitanie""]",#1 of 8 Restaurants in Occitanie,NaN
10064,La Bastide Cabezac,France,Occitanie,NaN,NaN,Occitanie,popularity_detailed_repair,"[""Europe"", ""France"", ""Occitanie""]",#3 of 8 Restaurants in Occitanie,NaN



Top city_filled ramase suspecte:


,count
city_filled,
London,20057
Berlin,6530
United Kingdom (UK),3503
Hamburg,3218
Brussels,2515
Edinburgh,2143
Glasgow,2027
Bucharest,2006
Bristol,1462


In [ ]:

list_like_cols = [
    "top_tags",
    "meals",
    "features",
    "special_diets",
    "awards",
    "cuisine",
    "cuisines",
]

for col in list_like_cols:
    if col in df_clean.columns:
        df_clean[f"{col}_list"] = df_clean[col].apply(parse_list_field)

        df_clean[f"{col}_text"] = df_clean[f"{col}_list"].apply(
            lambda xs: ", ".join(xs) if xs else "Unknown"
        )

        df_clean[f"{col}_norm_list"] = df_clean[f"{col}_list"].apply(
            lambda xs: [normalize_for_match(x) for x in xs if normalize_for_match(x)]
        )

preview_cols = [
    c
    for c in [
        "top_tags",
        "top_tags_list",
        "meals",
        "meals_list",
        "features",
        "features_list",
        "special_diets",
        "special_diets_list",
    ]
    if c in df_clean.columns
]

df_clean[preview_cols].head()

,top_tags,top_tags_list,meals,meals_list,features,features_list,special_diets,special_diets_list
0,"Cheap Eats, French","[Cheap Eats, French]","Lunch, Dinner","[Lunch, Dinner]","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","[Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service]",NaN,[]
1,Cheap Eats,[Cheap Eats],NaN,[],NaN,[],NaN,[]
2,"Cheap Eats, French, European","[Cheap Eats, French, European]","Dinner, Lunch, Drinks","[Dinner, Lunch, Drinks]","Reservations, Seating, Table Service, Wheelchair Accessible","[Reservations, Seating, Table Service, Wheelchair Accessible]",NaN,[]
3,"Cheap Eats, French","[Cheap Eats, French]","Lunch, Dinner","[Lunch, Dinner]","Reservations, Seating, Serves Alcohol, Table Service, Wheelchair Accessible","[Reservations, Seating, Serves Alcohol, Table Service, Wheelchair Accessible]",NaN,[]
4,"Mid-range, French","[Mid-range, French]","Lunch, Dinner","[Lunch, Dinner]","Reservations, Seating, Wheelchair Accessible, Table Service","[Reservations, Seating, Wheelchair Accessible, Table Service]",NaN,[]


In [ ]:

def infer_price_bucket(row):
    price = safe_value(row.get("price_level"), default="")
    price_range = safe_value(row.get("price_range"), default="")
    tags = " ".join(row.get("top_tags_norm_list", []))

    raw = f"{price} {price_range}".strip()
    compact = re.sub(r"\s+", "", raw.lower())

    if compact in {"€", "$", "£"}:
        return "cheap"

    if "€€€€" in compact or "$$$$" in compact or "££££" in compact:
        return "expensive"

    if "€€-€€€" in compact or "€€€" in compact or "€€" in compact:
        return "mid"

    if "$$-$$$" in compact or "$$$" in compact or "$$" in compact:
        return "mid"

    if "££-£££" in compact or "£££" in compact or "££" in compact:
        return "mid"

    if "cheap eats" in tags or "cheap" in tags or "budget" in tags:
        return "cheap"

    if "mid range" in tags or "mid-range" in tags or "moderate" in tags:
        return "mid"

    if "fine dining" in tags or "luxury" in tags or "expensive" in tags:
        return "expensive"

    return "unknown"


df_clean["price_level"] = df_clean["price_level"].apply(clean_text_basic)
df_clean["price_bucket"] = df_clean.apply(infer_price_bucket, axis=1)
df_clean["has_price"] = df_clean["price_bucket"] != "unknown"

print("price_level:")
print(df_clean["price_level"].value_counts(dropna=False).head(20))

print("\nprice_bucket:")
print(df_clean["price_bucket"].value_counts(dropna=False))

price_level:
price_level
€€-€€€    537918
NaN       277205
€         240205
€€€€       28069
Name: count, dtype: int64

price_bucket:
price_bucket
mid          388769
unknown      276741
expensive    244921
cheap        172966
Name: count, dtype: int64


In [ ]:

def extract_popularity_rank_total(x):
    s = clean_text_basic(x)

    if pd.isna(s):
        return np.nan, np.nan

    m = re.search(r"#?\s*([\d,]+)\s+of\s+([\d,]+)", str(s), flags=re.IGNORECASE)

    if not m:
        return np.nan, np.nan

    rank = float(m.group(1).replace(",", ""))
    total = float(m.group(2).replace(",", ""))

    return rank, total


popularity_values = df_clean["popularity_detailed"].apply(extract_popularity_rank_total)

df_clean["popularity_rank_num"] = popularity_values.apply(lambda x: x[0])
df_clean["popularity_total_num"] = popularity_values.apply(lambda x: x[1])


def compute_popularity_score(row):
    rank = row.get("popularity_rank_num")
    total = row.get("popularity_total_num")

    if pd.isna(rank) or pd.isna(total) or total <= 1 or rank < 1:
        return np.nan

    score = 1 - (math.log1p(rank) / math.log1p(total))

    return float(np.clip(score, 0, 1))


df_clean["popularity_score"] = df_clean.apply(compute_popularity_score, axis=1)

display(
    df_clean[
        [
            "popularity_detailed",
            "popularity_rank_num",
            "popularity_total_num",
            "popularity_score",
        ]
    ].head(10)
)

display(df_clean["popularity_score"].describe())

,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score
0,#1 of 2 Restaurants in Saint-Jouvent,1.0,2.0,0.36907
1,#2 of 2 Restaurants in Saint-Jouvent,2.0,2.0,0.00000
2,#1 of 1 Restaurant in Rivarennes,1.0,1.0,NaN
3,#1 of 1 Restaurant in Lacelle,1.0,1.0,NaN
4,#1 of 1 Restaurant in Saint-Laurent-de-Levezou,1.0,1.0,NaN
5,#1 of 1 Restaurant in Le Crozet,1.0,1.0,NaN
6,#2 of 2 Restaurants in Saint-Denis,2.0,2.0,0.00000
7,#1 of 2 Restaurants in Saint-Denis,1.0,2.0,0.36907
8,#1 of 1 Restaurant in Orgibet,1.0,1.0,NaN
9,#1 of 1 Restaurant in They-sous-Montfort,1.0,1.0,NaN


,popularity_score
count,952836.000000
mean,0.197928
std,0.188069
min,0.000000
25%,0.051109
50%,0.138647
75%,0.296642
max,0.928852


In [ ]:

essential_cols = ["restaurant_id", "name", "country"]

before = len(df_clean)
df_clean = df_clean.dropna(
    subset=[c for c in essential_cols if c in df_clean.columns]
).reset_index(drop=True)
after = len(df_clean)

print(f"Eliminate {before - after} randuri fara campuri esentiale.")
print("Shape nou:", df_clean.shape)

Eliminate 3 randuri fara campuri esentiale.
Shape nou: (1083394, 61)


In [ ]:

df_clean["has_city"] = df_clean["city_filled"].notna() & (~df_clean["city_is_suspicious"])
df_clean["has_country"] = df_clean["country"].notna()
df_clean["has_region"] = df_clean["region"].notna()
df_clean["has_address"] = df_clean["address"].notna()
df_clean["has_rating"] = df_clean["rating"].notna()
df_clean["has_popularity"] = df_clean["popularity_detailed"].notna()
df_clean["has_tags"] = (
    df_clean["top_tags_list"].apply(len) > 0
    if "top_tags_list" in df_clean.columns
    else False
)
df_clean["has_meals"] = (
    df_clean["meals_list"].apply(len) > 0
    if "meals_list" in df_clean.columns
    else False
)
df_clean["has_features"] = (
    df_clean["features_list"].apply(len) > 0
    if "features_list" in df_clean.columns
    else False
)
df_clean["has_special_diets"] = (
    df_clean["special_diets_list"].apply(len) > 0
    if "special_diets_list" in df_clean.columns
    else False
)

df_clean["profile_quality_score"] = (
    df_clean["has_city"].astype(int) * 2
    + df_clean["has_country"].astype(int)
    + df_clean["has_region"].astype(int)
    + df_clean["has_address"].astype(int)
    + df_clean["has_rating"].astype(int)
    + df_clean["has_price"].astype(int)
    + df_clean["has_popularity"].astype(int)
    + df_clean["has_tags"].astype(int)
    + df_clean["has_meals"].astype(int)
    + df_clean["has_features"].astype(int)
    + df_clean["has_special_diets"].astype(int)
)

display(
    df_clean[
        [
            "has_city",
            "has_country",
            "has_region",
            "has_address",
            "has_rating",
            "has_price",
            "has_popularity",
            "has_tags",
            "has_meals",
            "has_features",
            "has_special_diets",
            "profile_quality_score",
        ]
    ].head()
)

display(df_clean["profile_quality_score"].describe())

,has_city,has_country,has_region,has_address,has_rating,has_price,has_popularity,has_tags,has_meals,has_features,has_special_diets,profile_quality_score
0,True,True,True,True,True,True,True,True,True,True,False,11
1,True,True,True,True,True,True,True,True,False,False,False,9
2,True,True,True,True,True,True,True,True,True,True,False,11
3,True,True,True,True,True,True,True,True,True,True,False,11
4,True,True,True,True,True,True,True,True,True,True,False,11


,profile_quality_score
count,1.083394e+06
mean,9.522441e+00
std,1.647228e+00
min,2.000000e+00
25%,8.000000e+00
50%,1.000000e+01
75%,1.100000e+01
max,1.200000e+01


In [ ]:

MIN_PROFILE_SCORE_FOR_INDEX = 5

df_indexable = df_clean[
    (df_clean["has_city"])
    & (df_clean["profile_quality_score"] >= MIN_PROFILE_SCORE_FOR_INDEX)
].copy().reset_index(drop=True)

print("Restaurante procesate:", len(df_clean))
print("Restaurante indexabile:", len(df_indexable))
print("Procent indexabil:", round(len(df_indexable) / len(df_clean) * 100, 2), "%")

display(df_indexable["profile_quality_score"].describe())

Restaurante procesate: 1083394
Restaurante indexabile: 1033881
Procent indexabil: 95.43 %


,profile_quality_score
count,1.033881e+06
mean,9.618651e+00
std,1.579935e+00
min,5.000000e+00
25%,9.000000e+00
50%,1.000000e+01
75%,1.100000e+01
max,1.200000e+01


In [ ]:

def build_restaurant_profile(row):
    name = safe_value(row.get("name"))
    country = safe_value(row.get("country"))
    region = safe_value(row.get("region"))
    province = safe_value(row.get("province"))
    city = safe_value(row.get("city_filled"))
    address = safe_value(row.get("address"))

    rating = row.get("rating")
    price_level = safe_value(row.get("price_level"))
    price_bucket = safe_value(row.get("price_bucket"))

    top_tags = safe_value(row.get("top_tags_text"))
    meals = safe_value(row.get("meals_text"))
    features = safe_value(row.get("features_text"))
    special_diets = safe_value(row.get("special_diets_text"))
    popularity_detailed = safe_value(row.get("popularity_detailed"))

    location_parts = []

    for x in [city, province, region, country]:
        if x != "Unknown" and x not in location_parts:
            location_parts.append(x)

    location_text = ", ".join(location_parts)

    parts = []

    if location_text:
        parts.append(f"{name} is a restaurant located in {location_text}.")
    else:
        parts.append(f"{name} is a restaurant.")

    if address != "Unknown":
        parts.append(f"Address: {address}.")

    if pd.notna(rating):
        parts.append(f"Average rating: {rating} out of 5.")

    if price_bucket != "Unknown" and price_bucket != "unknown":
        parts.append(f"Price category: {price_bucket}.")

    if price_level != "Unknown":
        parts.append(f"Original price level: {price_level}.")

    if top_tags != "Unknown":
        parts.append(f"Cuisines and tags: {top_tags}.")

    if special_diets != "Unknown":
        parts.append(f"Special diets: {special_diets}.")

    if meals != "Unknown":
        parts.append(f"Meals served: {meals}.")

    if features != "Unknown":
        parts.append(f"Features: {features}.")

    if popularity_detailed != "Unknown":
        parts.append(f"Popularity: {popularity_detailed}.")

    return " ".join(parts)


def build_short_profile(row):
    name = safe_value(row.get("name"))
    city = safe_value(row.get("city_filled"))
    country = safe_value(row.get("country"))
    rating = row.get("rating")
    price_bucket = safe_value(row.get("price_bucket"))
    tags = safe_value(row.get("top_tags_text"))

    text = f"{name} | {city}, {country}"

    if pd.notna(rating):
        text += f" | rating={rating}"

    if price_bucket != "Unknown" and price_bucket != "unknown":
        text += f" | price={price_bucket}"

    if tags != "Unknown":
        text += f" | tags={tags}"

    return text

In [ ]:

df_clean["profile_text"] = df_clean.apply(build_restaurant_profile, axis=1)
df_clean["short_profile"] = df_clean.apply(build_short_profile, axis=1)

df_indexable["profile_text"] = df_indexable.apply(build_restaurant_profile, axis=1)
df_indexable["short_profile"] = df_indexable.apply(build_short_profile, axis=1)

display(df_indexable[["restaurant_id", "name", "short_profile", "profile_text"]].head(5))

,restaurant_id,name,short_profile,profile_text
0,g10001637-d10002227,Le 147,"Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French","Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Featu..."
1,g10001637-d14975787,Le Saint Jouvent,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats","Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaur..."
2,g10002858-d4586832,Au Bout du Pont,"Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European","Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch..."
3,g10002986-d3510044,Le Relais de Naiade,"Le Relais de Naiade | Lacelle, France | rating=4.0 | price=cheap | tags=Cheap Eats, French","Le Relais de Naiade is a restaurant located in Lacelle, Correze, Nouvelle-Aquitaine, France. Address: 9 avenue Porte de la Correze 19170, 19170 Lacelle France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch..."
4,g10022428-d9767191,Relais Du MontSeigne,"Relais Du MontSeigne | Saint-Laurent-de-Levezou, France | rating=4.5 | price=mid | tags=Mid-range, French","Relais Du MontSeigne is a restaurant located in Saint-Laurent-de-Levezou, Aveyron, Occitanie, France. Address: route du Montseigne, 12620 Saint-Laurent-de-Levezou France. Average rating: 4.5 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, French. Meals ..."


In [ ]:

def contains_bad_nan_text(s):
    if pd.isna(s):
        return True

    return bool(re.search(r"\b(nan|none|null)\b", str(s), flags=re.IGNORECASE))


bad_profiles = df_indexable["profile_text"].apply(contains_bad_nan_text)

print("Profile indexabile cu nan/none/null textual:", int(bad_profiles.sum()))

if bad_profiles.sum() > 0:
    display(df_indexable.loc[bad_profiles, ["name", "profile_text"]].head(20))

print("\nTop tari:")
display(df_indexable["country"].value_counts(dropna=False).head(15))

print("\nTop orase completate:")
display(df_indexable["city_filled"].value_counts(dropna=False).head(20))

print("\nPrice buckets:")
display(df_indexable["price_bucket"].value_counts(dropna=False))

print("\nCity sources:")
display(df_indexable["city_source"].value_counts(dropna=False))

Profile indexabile cu nan/none/null textual: 83


,name,profile_text
6848,O Nan House,"O Nan House is a restaurant located in Cergy-Pontoise, Val-d'Oise, Ile-de-France, France. Address: 221 rue Henri Dunant, 95300, Cergy-Pontoise France. Average rating: 3.0 out of 5. Cuisines and tags: Fast food. Meals served: Lunch, Dinner. Popularity: #11 of 19 Restaurants in Cergy-Pontoise."
22654,Chez Yves,"Chez Yves is a restaurant located in Cantobre, Aveyron, Occitanie, France. Address: None, Cantobre France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Breakfast, Lunch, Dinner. Features: Reservations, Outdoor ..."
26459,Pizza Nan's,"Pizza Nan's is a restaurant located in Saint-Marcellin, Isere, Auvergne-Rhone-Alpes, France. Address: 6 B avenue de Romans, 38160 Saint-Marcellin France. Average rating: 4.0 out of 5. Meals served: Lunch, Dinner. Popularity: #10 of 11 Restaurants in Saint-Marcellin."
36312,Nan Kebab,"Nan Kebab is a restaurant located in Tours, Indre-et-Loire, Centre-Val de Loire, France. Address: 14 avenue Andre Maginot, 37100, Tours France. Average rating: 2.5 out of 5. Price category: mid. Original price level: €. Cuisines and tags: Cheap Eats, Middle Eastern. Popularity: #414 of 432 Resta..."
38022,Deli nan Hassan,"Deli nan Hassan is a restaurant located in Besancon, Doubs, Bourgogne-Franche-Comte, France. Address: 90 rue Battant, 25000, Besancon France. Average rating: 3.5 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, Middle Eastern. Meals served: Dinner. Featu..."
38106,Deli' Nan 2,"Deli' Nan 2 is a restaurant located in Besancon, Doubs, Bourgogne-Franche-Comte, France. Address: 31 rue d Arenes, 25000, Besancon France."
38711,Qiao Jiang Nan,"Qiao Jiang Nan is a restaurant located in Paris, Ile-de-France, France. Address: 43 rue de Provence, 75009 Paris France. Average rating: 4.0 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, Chinese, Asian. Meals served: Dinner, Lunch. Features: Seating, ..."
46247,Nouilles De Jiang Nan,"Nouilles De Jiang Nan is a restaurant located in Paris, Ile-de-France, France. Address: 21 rue Mademoiselle, 75015 Paris France. Average rating: 4.5 out of 5. Meals served: Lunch, Dinner. Features: Reservations. Popularity: #10388 of 15476 Restaurants in Paris."
51958,Jiang Nan,"Jiang Nan is a restaurant located in Paris, Ile-de-France, France. Address: 10 rue de Mazagran metro Bonne-Nouvelle ou Strasbourg St-denis, 75010 Paris France. Average rating: 4.5 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, Chinese, Asian, Vietnamese...."
57473,Le Cheese Nan,"Le Cheese Nan is a restaurant located in Montpellier, Herault, Occitanie, France. Address: 17 Rue de Verdun, 2 rue Aristide Olivier, 34000, Montpellier France. Average rating: 2.0 out of 5. Price category: mid. Original price level: €. Cuisines and tags: Cheap Eats, Indian, Fast food, Turkish. M..."



Top tari:


,count
country,
Italy,224476
Spain,156870
France,155136
England,119241
Germany,104477
Greece,33753
Portugal,32587
The Netherlands,29422
Poland,24689



Top orase completate:


,count
city_filled,
Paris,18129
Rome,12603
Madrid,12134
Barcelona,10285
Milan,8382
Prague,6035
Lisbon,5261
Vienna,4571
Amsterdam,4325



Price buckets:


,count
price_bucket,
mid,372580
unknown,262779
expensive,233064
cheap,165458



City sources:


,count
city_source,
original_city,681400
original_location,352465
popularity_detailed,16


In [ ]:

sample_cols = [
    "restaurant_id",
    "name",
    "country",
    "region",
    "province",
    "city",
    "city_filled",
    "city_source",
    "city_is_suspicious",
    "address",
    "rating",
    "price_level",
    "price_bucket",
    "top_tags_text",
    "special_diets_text",
    "meals_text",
    "features_text",
    "popularity_detailed",
    "popularity_rank_num",
    "popularity_total_num",
    "popularity_score",
    "profile_quality_score",
    "profile_text",
]

sample_cols = [c for c in sample_cols if c in df_indexable.columns]

display(
    df_indexable[sample_cols].sample(
        min(10, len(df_indexable)),
        random_state=42,
    )
)

,restaurant_id,name,country,region,province,city,city_filled,city_source,city_is_suspicious,address,rating,price_level,price_bucket,top_tags_text,special_diets_text,meals_text,features_text,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,profile_text
232122,g187375-d15052680,Cafe Lebenslust,Germany,North Rhine-Westphalia,NaN,Essen,Essen,original_city,False,"Limbecker Platz 1a Shopping Centre Limbecker Platz, 45127 Essen, North Rhine-Westphalia Germany",3.5,NaN,unknown,"Italian, Cafe, Street Food",Unknown,Unknown,Unknown,#21 of 22 Coffee & Tea in Essen,21.0,22.0,0.014177,8,"Cafe Lebenslust is a restaurant located in Essen, North Rhine-Westphalia, Germany. Address: Limbecker Platz 1a Shopping Centre Limbecker Platz, 45127 Essen, North Rhine-Westphalia Germany. Average rating: 3.5 out of 5. Cuisines and tags: Italian, Cafe, Street Food. Popularity: #21 of 22 Coffee &..."
619281,g1069546-d2471237,La baita di Campotenese,Italy,Calabria,Province of Cosenza,NaN,Morano Calabro,original_location,False,"C.da Campotenese, Morano Calabro Italy",4.0,€€-€€€,mid,"Mid-range, Italian, Pizza, Mediterranean",Unknown,"Breakfast, Lunch, Dinner","Reservations, Seating, Serves Alcohol, Table Service, Takeout",#9 of 9 Restaurants in Morano Calabro,9.0,9.0,0.000000,11,"La baita di Campotenese is a restaurant located in Morano Calabro, Province of Cosenza, Calabria, Italy. Address: C.da Campotenese, Morano Calabro Italy. Average rating: 4.0 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, Italian, Pizza, Mediterranean. ..."
195858,g1085669-d19911015,Hanoi's Ecke,Germany,Hesse,NaN,Ober-Moerlen,Ober-Moerlen,original_city,False,"Frankfurter Str. 37, 61239 Ober-Moerlen, Hesse Germany",5.0,€€-€€€,mid,"Mid-range, Sushi, Vietnamese",Unknown,Unknown,Unknown,#1 of 6 Restaurants in Ober-Moerlen,1.0,6.0,0.643793,9,"Hanoi's Ecke is a restaurant located in Ober-Moerlen, Hesse, Germany. Address: Frankfurter Str. 37, 61239 Ober-Moerlen, Hesse Germany. Average rating: 5.0 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, Sushi, Vietnamese. Popularity: #1 of 6 Restaurants..."
1435,g1025646-d3469776,Le Bacchus,France,Occitanie,Aude,Port La Nouvelle,Port La Nouvelle,original_city,False,"346 rue Georges Clemenceau, 11210 Port La Nouvelle France",4.5,€€-€€€,mid,"Mid-range, French, Wine Bar",Unknown,"Lunch, Dinner, Drinks, Breakfast","Reservations, Outdoor Seating, Seating, Wheelchair Accessible, Serves Alcohol, Table Service",#16 of 26 Restaurants in Port La Nouvelle,16.0,26.0,0.140366,11,"Le Bacchus is a restaurant located in Port La Nouvelle, Aude, Occitanie, France. Address: 346 rue Georges Clemenceau, 11210 Port La Nouvelle France. Average rating: 4.5 out of 5. Price category: mid. Original price level: €€-€€€. Cuisines and tags: Mid-range, French, Wine Bar. Meals served: Lunc..."
413040,g2359638-d8808423,Restaurante Vida Gastrobar,Spain,Extremadura,Province of Caceres,NaN,Villanueva de la Vera,original_location,False,"Avenida de la Vera 57, 10470 Villanueva de la Vera Spain",4.5,€€-€€€,expensive,"Mid-range, Mediterranean, Spanish, Diner","Vegetarian Friendly, Vegan Options, Gluten Free Options",Unknown,Unknown,#3 of 8 Restaurants in Villanueva de la Vera,3.0,8.0,0.369070,10,"Restaurante Vida Gastrobar is a restaurant located in Villanueva de la Vera, Province of Caceres, Extremadura, Spain. Address: Avenida de la Vera 57, 10470 Villanueva de la Vera Spain. Average rating: 4.5 out of 5. Price category: expensive. Original price level: €€-€€€. Cuisines and tags: Mid-r..."
826976,g799534-d3498234,Pasqualotto,Italy,Lazio,Province of Rome,NaN,Marino,original_location,False,"Piazza Giacomo Matteotti 11, 00040, Marino Italy",4.5,€,cheap,"Cheap Eats, Dessert",Unknown,Unknown,"Takeout, Seating, Wheelchair Accessible",#2 of 4 Dessert Spots in Marino,2.0,4.0,0.317394,10,"Pasqualotto is a restaurant located in Marino, Province of Rome, Lazio, Ital

In [ ]:

output_dir = Path("/content/drive/MyDrive/tablewise/data_new/processed")
output_dir.mkdir(parents=True, exist_ok=True)

processed_csv = output_dir / "restaurants_processed.csv"
processed_parquet = output_dir / "restaurants_processed.parquet"
indexable_csv = output_dir / "restaurants_indexable.csv"
indexable_parquet = output_dir / "restaurants_indexable.parquet"

df_clean.to_csv(processed_csv, index=False)
df_clean.to_parquet(processed_parquet, index=False)

df_indexable.to_csv(indexable_csv, index=False)
df_indexable.to_parquet(indexable_parquet, index=False)

print("Saved processed CSV:", processed_csv)
print("Saved processed Parquet:", processed_parquet)
print("Saved indexable CSV:", indexable_csv)
print("Saved indexable Parquet:", indexable_parquet)

Saved processed CSV: /content/drive/MyDrive/tablewise/data_new/processed/restaurants_processed.csv
Saved processed Parquet: /content/drive/MyDrive/tablewise/data_new/processed/restaurants_processed.parquet
Saved indexable CSV: /content/drive/MyDrive/tablewise/data_new/processed/restaurants_indexable.csv
Saved indexable Parquet: /content/drive/MyDrive/tablewise/data_new/processed/restaurants_indexable.parquet
